In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-8342900666925943>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import *
      3 from pyspark.sql.types import *

ModuleNotFoundError: No module named 'dlt'

In [0]:
@dlt.view

def silver_employee_cdf_type2_stage():
    df = spark.readStream.table("employee_cdf_type2_stage")
    # Access correct column names (two underscores)
    df = df.withColumn("start_dt", col("__START_AT"))
    df = df.withColumn("end_dt", col("__END_AT"))
    # Add is_active and load_dt columns
    df = df.withColumn(
        "is_active", 
        when(col("end_dt").isNull(), lit("Y")).otherwise(lit("N"))
    )
    df = df.withColumn("load_dt", current_timestamp())
    # Drop internal CDC tracking columns
    return df.drop("__START_AT", "__END_AT")

In [0]:
dlt.create_streaming_table("gold_employee_cdf_streaming_type2")
dlt.apply_changes(
    target = "gold_employee_cdf_streaming_type2",
    source = "silver_employee_cdf_type2_stage",
    keys = ["Employee_ID"],
    sequence_by = struct("Employee_ID", "_commit_timestamp"),
    ignore_null_updates = True,  # columns with a null retain existing values in the target during update
    apply_as_deletes = expr("_change_type = 'delete'"),
    except_column_list = ["_change_type", "_commit_version"],
    stored_as_scd_type = 1
)